# 02 — Classical ML: HOG + Color Histogram + GLCM → PCA → RBF SVM

HOG is computed on **128×128** images to keep the descriptor manageable.
PCA reduces ~3.5k → 200 dims before the SVM.

In [1]:
# === Colab preamble: clone repo, mount Drive, set up paths ===
import os, sys, subprocess

REPO_URL  = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
REPO_PATH = "/content/melanoma-detection-ham10000"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

# Mount Drive (silently re-uses existing mount on re-run)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import config
config.ensure_drive_dirs()
print("Drive root :", config.DRIVE_ROOT)
print("Data dir   :", config.DATA_DIR)
print("Results dir:", config.RESULTS_DIR)

Mounted at /content/drive
Drive root : /content/drive/MyDrive/melanoma
Data dir   : /content/drive/MyDrive/melanoma/data
Results dir: /content/drive/MyDrive/melanoma/results


In [2]:
import random, numpy as np, torch
random.seed(config.SEED); np.random.seed(config.SEED); torch.manual_seed(config.SEED)
torch.cuda.manual_seed_all(config.SEED)

In [3]:
import numpy as np, cv2
from tqdm.auto import tqdm
from src.data import load_arrays_balanced
from src.features import extract_all
X, y, ids, idx_train, idx_val, idx_test = load_arrays_balanced(config.DATA_DIR)
print("BALANCED test set -> n:", len(idx_test),
      " mel/non-mel:", int(y[idx_test].sum()), "/", int((y[idx_test]==0).sum()))
print("X:", X.shape)

X: (10015, 224, 224, 3)


In [4]:
# --- Resize once to 128x128 (HOG is RAM-hungry on 224x224) ---
def to_hog_size(X224, size):
    out = np.empty((len(X224), size, size, 3), dtype=np.uint8)
    for i, im in enumerate(X224):
        out[i] = cv2.resize(im, (size, size), interpolation=cv2.INTER_AREA)
    return out

X128 = to_hog_size(X, config.HOG_IMG_SIZE)
print("Resized:", X128.shape)

Resized: (10015, 128, 128, 3)


In [5]:
# --- Extract concatenated descriptors ---
sample = extract_all(X128[0],
                     pixels_per_cell=config.HOG_PIXELS_PER_CELL,
                     cells_per_block=config.HOG_CELLS_PER_BLOCK,
                     color_bins=config.COLOR_HIST_BINS,
                     glcm_distances=config.GLCM_DISTANCES,
                     glcm_angles=config.GLCM_ANGLES)
print("Feature dim:", sample.shape[0])

F = np.empty((len(X128), sample.shape[0]), dtype=np.float32)
for i in tqdm(range(len(X128)), desc="features"):
    F[i] = extract_all(X128[i],
                       pixels_per_cell=config.HOG_PIXELS_PER_CELL,
                       cells_per_block=config.HOG_CELLS_PER_BLOCK,
                       color_bins=config.COLOR_HIST_BINS,
                       glcm_distances=config.GLCM_DISTANCES,
                       glcm_angles=config.GLCM_ANGLES)
print("F:", F.shape, F.dtype)

Feature dim: 2292


features:   0%|          | 0/10015 [00:00<?, ?it/s]

F: (10015, 2292) float32


In [6]:
# --- Standardize + PCA (fit on train only) ---
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler().fit(F[idx_train])
F_s = scaler.transform(F)

pca = PCA(n_components=config.PCA_COMPONENTS, random_state=config.SEED).fit(F_s[idx_train])
F_p = pca.transform(F_s)
print("PCA explained variance:", pca.explained_variance_ratio_.sum().round(3))
print("F_p:", F_p.shape)

PCA explained variance: 0.746
F_p: (10015, 200)


In [7]:
# --- Train RBF SVM with class_weight='balanced' ---
import time
from sklearn.svm import SVC
hp = dict(kernel="rbf", C=1.0, gamma="scale",
          class_weight="balanced", probability=True, random_state=config.SEED)
svm = SVC(**hp)
t0 = time.time(); svm.fit(F_p[idx_train], y[idx_train]); train_time = time.time() - t0
print(f"Trained in {train_time:.1f}s")

Trained in 31.8s


In [8]:
# --- Evaluate ---
from src.evaluation import save_standard_outputs, time_inference

y_pred = svm.predict(F_p[idx_test])
y_prob = svm.predict_proba(F_p[idx_test])[:, 1]

inf_ms = time_inference(lambda x: svm.predict(x), F_p[idx_test][:200])

metrics = save_standard_outputs(
    method_name="classical_ml_svm",
    results_dir=config.RESULTS_DIR,
    y_true=y[idx_test], y_pred=y_pred, y_prob=y_prob,
    ids=ids[idx_test],
    hyperparameters={**hp, "feature_dim_raw": int(F.shape[1]), "pca_components": config.PCA_COMPONENTS},
    train_time_sec=train_time,
    inference_time_per_image_ms=inf_ms,
)
{k: v for k, v in metrics.items() if k != "hyperparameters"}

{'accuracy': 0.8396540252827678,
 'precision': 0.3576923076923077,
 'recall': 0.5568862275449101,
 'f1': 0.43559718969555034,
 'roc_auc': 0.8350223206282046,
 'train_time_sec': 31.75062656402588,
 'inference_time_per_image_ms': 1.1260626550006236}